In [1]:
import pickle
import torch
import numpy as np
from tqdm.notebook import tqdm
import pandas as pd
import sys

# Add paths so Python can find the event_log_loader module when unpickling
sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')

In [2]:
# Paths: perturbed file (event log format, possibly with many columns) and Weytjens encoder
data_path = '../../../../perturbed_data/helpdesk/val.pkl'
result_path = '../../../../encoded_data/weytjens/helpdesk/val_new.pkl'
encoder_path = '../../../../encoded_data/data_encoder/weytjens_helpdesk_encoder_decoder.pkl'
properties_path = '../../../../encoded_data/data_encoder/weytjens_helpdesk_event_log_properties.pkl'

# Weytjens feature set (same as Helpdesk_full_loader): only Activity + case_elapsed_time
WEYTJENS_CASE_NAME = 'Case ID'
WEYTJENS_CATEGORICAL_COLUMNS = ['Activity']
WEYTJENS_CONTINUOUS_COLUMNS = ['case_elapsed_time']

In [3]:
def reduce_to_weytjens_features(df):
    """Keep only Weytjens columns (Case ID, Activity, case_elapsed_time). Missing cols filled with 0 or 'UNK'."""
    keep = [WEYTJENS_CASE_NAME] + WEYTJENS_CATEGORICAL_COLUMNS + WEYTJENS_CONTINUOUS_COLUMNS
    out = df.copy()
    for c in keep:
        if c not in out.columns:
            if c in WEYTJENS_CONTINUOUS_COLUMNS:
                out[c] = 0.0
            elif c in WEYTJENS_CATEGORICAL_COLUMNS:
                out[c] = 'UNK'
    return out[keep]

In [4]:
# Load perturbed data (event log format: dict (case_id, prefix_len) -> (prefix_df, suffix_df))
data = torch.load(data_path, weights_only=False)
print(f"Loaded {len(data)} prefix/suffix pairs")

# Reduce to Weytjens features only
reduced_data = {}
for (case_id, prefix_len), (prefix_df, suffix_df) in tqdm(data.items(), desc="Reducing to Weytjens features"):
    prefix_reduced = reduce_to_weytjens_features(prefix_df)
    suffix_reduced = reduce_to_weytjens_features(suffix_df)
    reduced_data[(case_id, prefix_len)] = (prefix_reduced, suffix_reduced)

print(f"Reduced {len(reduced_data)} pairs to Weytjens features (Activity, case_elapsed_time)")

Loaded 1898 prefix/suffix pairs


Reducing to Weytjens features:   0%|          | 0/1898 [00:00<?, ?it/s]

Reduced 1898 pairs to Weytjens features (Activity, case_elapsed_time)


In [5]:
def encode_single_dataframe(df, encoder_decoder, case_name_col, case_id_value):
    """
    Encode a single DataFrame (prefix or suffix) into tensor format.
    """
    df_copy = df.copy()
    if case_name_col not in df_copy.columns:
        df_copy[case_name_col] = case_id_value
    else:
        df_copy[case_name_col] = case_id_value
    
    cat_tensors = []
    for col in encoder_decoder.categorical_columns:
        if col not in df_copy.columns:
            cat_tensors.append(torch.zeros((1, encoder_decoder.window_size), dtype=torch.long))
            continue
        case_values = np.array(df_copy[[col]], dtype=object)
        case_values_enc = encoder_decoder.categorical_encoders[col].transform(case_values) + 1
        padded = encoder_decoder.pad_to_window_size(case_values_enc)
        tensor = torch.tensor(padded, dtype=torch.long).squeeze(-1).unsqueeze(0)
        cat_tensors.append(tensor)
    
    num_tensors = []
    for col in encoder_decoder.continuous_columns + encoder_decoder.continuous_positive_columns:
        if col not in df_copy.columns:
            num_tensors.append(torch.zeros((1, encoder_decoder.window_size), dtype=torch.float32))
            continue
        case_values = df_copy[[col]].values
        case_values_imputed = encoder_decoder.continuous_imputers[col].transform(case_values)
        case_values_enc = encoder_decoder.continuous_encoders[col].transform(case_values_imputed)
        padded = encoder_decoder.pad_to_window_size(case_values_enc)
        tensor = torch.tensor(padded, dtype=torch.float32).squeeze(-1).unsqueeze(0)
        num_tensors.append(tensor)
    
    return (cat_tensors, num_tensors)

In [6]:
# Load Weytjens encoder and event log properties
encoder_decoder = torch.load(encoder_path, weights_only=False)
with open(properties_path, 'rb') as f:
    props = pickle.load(f)

# Encode all prefix/suffix pairs
encoded_data = {}
for (case_id, prefix_len), (prefix_df, suffix_df) in tqdm(reduced_data.items(), desc="Encoding data"):
    encoded_prefix = encode_single_dataframe(
        prefix_df, encoder_decoder, props["case_name"], case_id
    )
    encoded_suffix = encode_single_dataframe(
        suffix_df, encoder_decoder, props["case_name"], case_id
    )
    encoded_data[(case_id, prefix_len)] = (encoded_prefix, encoded_suffix)

print(f"Encoded {len(encoded_data)} prefix/suffix pairs")
torch.save(encoded_data, result_path)
print(f"Saved to {result_path}")

Encoding data:   0%|          | 0/1898 [00:00<?, ?it/s]

Encoded 1898 prefix/suffix pairs
Saved to ../../../../encoded_data/weytjens/helpdesk/val_new.pkl
